# ETL da camada silver para camada gold


## Extract

In [1]:
import pandas as pd
import psycopg
from psycopg import connect, sql
import sys
import warnings

warnings.filterwarnings('ignore')

print("--- Iniciando processo de Extract do banco ---")

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA = "silver"
TABLE_NAME = "listings"

TABLE_FULL_NAME = sql.SQL("{}.{}").format(
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(TABLE_NAME)
)

query_object = sql.SQL("SELECT * FROM {}").format(TABLE_FULL_NAME)

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

try:
    print("Estabelecendo conexão...")
    
    with connect(connection_string) as conn:
        print("Conexão estabelecida.")
        
        query_string = query_object.as_string(conn)
        print(f"Executando query: {query_string}")

        df = pd.read_sql_query(query_string, conn)

    print("\nDados carregados do banco para o DataFrame com sucesso!")
    print(f"Total de linhas carregadas: {len(df)}")
except psycopg.Error as e:
    print(f"\n--- Ocorreu um erro ao conectar ou ler o banco de dados ---")
    print(f"Erro: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\n--- Ocorreu um erro inesperado ---")
    print(f"Erro: {e}")
    sys.exit(1)

--- Iniciando processo de Extract do banco ---
Estabelecendo conexão...
Conexão estabelecida.
Executando query: SELECT * FROM "silver"."listings"

Dados carregados do banco para o DataFrame com sucesso!
Total de linhas carregadas: 99417


In [2]:
df.head(3)

,id,host_id,name,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,1001254,80014485718,Clean & quiet apt home by the park,False,Madaline,Brooklyn,Kensington,40.64749,-73.97237,False,...,193.0,10,9,2021-10-19,0.21,4.0,6,286,Clean up and treat the home the way you'd like...,True
1,1002102,52335172823,Skylit Midtown Castle,True,Jenna,Manhattan,Midtown,40.75362,-73.98377,False,...,28.0,30,45,2022-05-21,0.38,4.0,2,228,Pet friendly but please confirm with me if the...,True
2,1002403,78829239556,THE VILLAGE OF HARLEM....NEW YORK !,True,Elise,Manhattan,Harlem,40.80902,-73.94190,True,...,124.0,3,0,None,NaN,5.0,1,352,"I encourage you to use my kitchen, cooking and...",True


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99417 entries, 0 to 99416
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              99417 non-null  int64  
 1   host_id                         99417 non-null  int64  
 2   name                            99417 non-null  object 
 3   host_identity_verified          99417 non-null  bool   
 4   host_name                       99417 non-null  object 
 5   neighbourhood_group             99417 non-null  object 
 6   neighbourhood                   99417 non-null  object 
 7   lat                             99417 non-null  float64
 8   long                            99417 non-null  float64
 9   instant_bookable                99417 non-null  bool   
 10  cancellation_policy             99364 non-null  object 
 11  room_type                       99417 non-null  object 
 12  construction_year               

## Transform

In [4]:
mapa_colunas = {
    'id': 'id_anun',
    'host_id': 'id_anfi',

    # Dimensão Anfitrião
    'host_name': 'nm_anfi',
    'host_identity_verified': 'anfi_verif',
    'calculated_host_listings_count': 'anfi_tot_anun',

    # Dimensão Localização
    'neighbourhood_group': 'grp_bairro',
    'neighbourhood': 'bairro',
    'lat': 'lat',
    'long': 'long',

    # Dimensão Propriedade
    'name': 'nm_anun',
    'instant_bookable': 'res_inst',
    'cancellation_policy': 'pol_cancel',
    'room_type': 'tipo_qto',
    'construction_year': 'ano_constr',
    'minimum_nights': 'min_noites',
    'has_house_rules': 'tem_regras',
    'house_rules': 'regras_txt',

    # Dimensão Tempo
    'last_review': 'dt_ult_rev',

    # Fatos (Métricas)
    'price': 'preco',
    'service_fee': 'tx_serv',
    'number_of_reviews': 'tot_rev',
    'reviews_per_month': 'rev_mes',
    'review_rate_number': 'nota_rev',
    'availability_365': 'disp_365'
}

df = df.rename(columns=mapa_colunas)

df = df.drop(columns=['regras_txt'], errors='ignore')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99417 entries, 0 to 99416
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anun        99417 non-null  int64  
 1   id_anfi        99417 non-null  int64  
 2   nm_anun        99417 non-null  object 
 3   anfi_verif     99417 non-null  bool   
 4   nm_anfi        99417 non-null  object 
 5   grp_bairro     99417 non-null  object 
 6   bairro         99417 non-null  object 
 7   lat            99417 non-null  float64
 8   long           99417 non-null  float64
 9   res_inst       99417 non-null  bool   
 10  pol_cancel     99364 non-null  object 
 11  tipo_qto       99417 non-null  object 
 12  ano_constr     99417 non-null  int64  
 13  preco          99417 non-null  float64
 14  tx_serv        99417 non-null  float64
 15  min_noites     99417 non-null  int64  
 16  tot_rev        99417 non-null  int64  
 17  dt_ult_rev     84233 non-null  object 
 18  rev_me

## Load

In [6]:
import pandas as pd
import psycopg
from psycopg import connect, sql
import sys
import warnings

warnings.filterwarnings('ignore')

# --- 1. CONFIGURAÇÕES DO BANCO ---
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA_GOLD = "gold"

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

# --- 2. EXTRACT ---
# REMOVIDO. Assumindo que 'df' já existe.

# --- 3. TRANSFORM & LOAD (Camada Gold) ---

print("\n--- Iniciando processo de Transform & Load (Camada Gold) ---")

if 'df' not in locals():
    print("Erro: DataFrame 'df' não encontrado na memória.")
    sys.exit(1)

try:
    # Abrir DDL da Camada Gold
    try:
        ddl_gold = open('../../data_layer/gold/gold_ddl.sql').read()
    except FileNotFoundError:
        print("Erro: Arquivo 'gold_ddl.sql' não encontrado.")
        sys.exit(1)

    with connect(connection_string) as conn:
        with conn.cursor() as cur:
            
            # 3.1. Executar DDL
            print("Executando DDL da camada Gold (Limpando e recriando tabelas)...")
            cur.execute(ddl_gold)
            print("Schema 'gold' e tabelas recriados.")

            # 3.2. Carga DIM_ANFITRIAO
            print("Iniciando carga: DIM_ANFITRIAO")
            cols_anfi = ['nm_anfi', 'anfi_verif', 'anfi_tot_anun']
            df_anfi = df[cols_anfi].drop_duplicates()
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_ANFITRIAO (nm_anfi, anfi_verif, anfi_tot_anun) VALUES (%s, %s, %s)")
            data_tuples = [tuple(x) for x in df_anfi.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            
            # CORREÇÃO: Usar minúsculas no RETURNING
            cur.execute("INSERT INTO gold.DIM_ANFITRIAO (nm_anfi, anfi_verif, anfi_tot_anun) VALUES (NULL, NULL, NULL) RETURNING srk_anfi")
            unknown_anfi_key = cur.fetchone()[0]
            print(f"DIM_ANFITRIAO carregada: {len(df_anfi)} registros únicos + 1 (Desconhecido).")


            # 3.3. Carga DIM_LOCALIZACAO
            print("Iniciando carga: DIM_LOCALIZACAO")
            cols_loc = ['lat', 'long', 'bairro', 'grp_bairro']
            df_loc = df[cols_loc].drop_duplicates()
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_LOCALIZACAO (lat, long, bairro, grp_bairro) VALUES (%s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_loc.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            
            # CORREÇÃO: Usar minúsculas no RETURNING
            cur.execute("INSERT INTO gold.DIM_LOCALIZACAO (lat, long, bairro, grp_bairro) VALUES (NULL, NULL, NULL, NULL) RETURNING srk_local")
            unknown_loc_key = cur.fetchone()[0]
            print(f"DIM_LOCALIZACAO carregada: {len(df_loc)} registros únicos + 1 (Desconhecido).")

            # 3.4. Carga DIM_PROPRIEDADE
            print("Iniciando carga: DIM_PROPRIEDADE")
            cols_prop = ['nm_anun', 'tipo_qto', 'min_noites', 'pol_cancel', 'res_inst', 'ano_constr', 'tem_regras']
            df_prop = df[cols_prop].drop_duplicates()
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_PROPRIEDADE (nm_anun, tipo_qto, min_noites, pol_cancel, res_inst, ano_constr, tem_regras) VALUES (%s, %s, %s, %s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_prop.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            
            # CORREÇÃO: Usar minúsculas no RETURNING
            cur.execute("INSERT INTO gold.DIM_PROPRIEDADE (nm_anun, tipo_qto, min_noites, pol_cancel, res_inst, ano_constr, tem_regras) VALUES (NULL, NULL, NULL, NULL, NULL, NULL, NULL) RETURNING srk_prop")
            unknown_prop_key = cur.fetchone()[0]
            print(f"DIM_PROPRIEDADE carregada: {len(df_prop)} registros únicos + 1 (Desconhecido).")

            # 3.5. Carga DIM_ULTIMA_AVALIACAO
            print("Iniciando carga: DIM_ULTIMA_AVALIACAO")
            col_aval = 'dt_ult_rev'
            df[col_aval] = pd.to_datetime(df[col_aval]) 
            df_aval = df[[col_aval]].drop_duplicates().dropna()

            df_aval['ano'] = df_aval[col_aval].dt.year.astype('Int64')
            df_aval['mes'] = df_aval[col_aval].dt.month.astype('Int64')
            df_aval['trimestre'] = df_aval[col_aval].dt.quarter.astype('Int64')

            insert_query = sql.SQL("INSERT INTO gold.DIM_ULTIMA_AVALIACAO (dt_ult_rev, ano, mes, trimestre) VALUES (%s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_aval.to_numpy()]
            cur.executemany(insert_query, data_tuples)

            # CORREÇÃO: Usar minúsculas no RETURNING
            cur.execute("INSERT INTO gold.DIM_ULTIMA_AVALIACAO (dt_ult_rev, ano, mes, trimestre) VALUES (NULL, NULL, NULL, NULL) RETURNING srk_aval")
            unknown_aval_key = cur.fetchone()[0]
            print(f"DIM_ULTIMA_AVALIACAO carregada: {len(df_aval)} registros + 1 (Desconhecido).")

            # --- 4. MAPEAMENTO DE CHAVES (MERGE) ---
            
            print("\nIniciando mapeamento de Chaves Surrogadas...")
            df_anfi_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_ANFITRIAO", conn)
            df_loc_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_LOCALIZACAO", conn)
            df_prop_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_PROPRIEDADE", conn)
            df_aval_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_ULTIMA_AVALIACAO", conn)

            df_aval_com_chaves['dt_ult_rev'] = pd.to_datetime(df_aval_com_chaves['dt_ult_rev'])
            
            df = pd.merge(df, df_anfi_com_chaves.drop_duplicates(subset=cols_anfi), on=cols_anfi, how='left')
            df = pd.merge(df, df_loc_com_chaves.drop_duplicates(subset=cols_loc), on=cols_loc, how='left')
            df = pd.merge(df, df_prop_com_chaves.drop_duplicates(subset=cols_prop), on=cols_prop, how='left')
            df = pd.merge(df, df_aval_com_chaves.drop_duplicates(subset=[col_aval]), on=col_aval, how='left')
            
            # #######################################################
            # CORREÇÃO: Usar minúsculas para acessar as colunas do DataFrame
            # #######################################################
            df['srk_anfi'] = df['srk_anfi'].fillna(unknown_anfi_key).astype(int)
            df['srk_local'] = df['srk_local'].fillna(unknown_loc_key).astype(int) 
            df['srk_prop'] = df['srk_prop'].fillna(unknown_prop_key).astype(int)
            df['srk_aval'] = df['srk_aval'].fillna(unknown_aval_key).astype(int)
            
            print("Mapeamento de chaves concluído.")
            
            # --- 5. LOAD FATO_ANUNCIO ---
            
            print("Iniciando carga: FATO_ANUNCIO")
            
            # #######################################################
            # CORREÇÃO: Usar minúsculas na lista de colunas
            # #######################################################
            cols_fato = [
                'disp_365', 'preco', 'tx_serv', 'tot_rev', 
                'rev_mes', 'nota_rev', 
                
                'srk_anfi', 'srk_local', 
                'srk_aval', 'srk_prop'
            ]
            
            df_fato = df[cols_fato]
            
            # #######################################################
            # CORREÇÃO: Usar minúsculas na query SQL
            # #######################################################
            insert_query = sql.SQL("""
                INSERT INTO gold.FATO_ANUNCIO (
                    disp_365, preco, tx_serv, tot_rev, 
                    rev_mes, nota_rev, 
                    srk_anfi, srk_local, 
                    srk_aval, srk_prop
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """)
            
            data_tuples = [tuple(x) for x in df_fato.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            print(f"FATO_ANUNCIO carregada: {len(df_fato)} registros.")

            # Commit final
            conn.commit()
            print("\n--- Processo ETL para Camada Gold CONCLUÍDO com sucesso! ---")

except psycopg.Error as e:
    print(f"\n--- Ocorreu um erro de psycopg no LOAD ---")
    print(f"Erro: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\n--- Ocorreu um erro inesperado no LOAD ---")
    print(f"Erro: {e}")
    sys.exit(1)


--- Iniciando processo de Transform & Load (Camada Gold) ---
Executando DDL da camada Gold (Limpando e recriando tabelas)...
Schema 'gold' e tabelas recriados.
Iniciando carga: DIM_ANFITRIAO
DIM_ANFITRIAO carregada: 25735 registros únicos + 1 (Desconhecido).
Iniciando carga: DIM_LOCALIZACAO
DIM_LOCALIZACAO carregada: 65408 registros únicos + 1 (Desconhecido).
Iniciando carga: DIM_PROPRIEDADE
DIM_PROPRIEDADE carregada: 93536 registros únicos + 1 (Desconhecido).
Iniciando carga: DIM_ULTIMA_AVALIACAO
DIM_ULTIMA_AVALIACAO carregada: 2436 registros + 1 (Desconhecido).

Iniciando mapeamento de Chaves Surrogadas...
Mapeamento de chaves concluído.
Iniciando carga: FATO_ANUNCIO
FATO_ANUNCIO carregada: 99417 registros.

--- Processo ETL para Camada Gold CONCLUÍDO com sucesso! ---
